# Week 4 · Tasks 1.1 & 1.3 — Polyp-size-stratified results

**Input (read-only):** `results/per_image_results.csv` (frozen, tag `week3-frozen`).
**Outputs:** `results/week4/size_bin_assignment.csv`, `results/week4/size_stratified_summary.csv`, `results/week4/hd95_excluded_by_bin.csv`.

Bins come from `configs/eval.yaml` → `stratification.polyp_size_pct` (small <5 %, medium 5–15 %, large ≥15 % of image area). Bins are left-closed, so an area of exactly 5.0 % is *medium*.

Aggregation rule: first average over test images **within each run**, then average over the 3 seeds for U-Net brackets. The ± is the **seed std** (not an image-level CI; bootstrap CIs are Sebastian's Week 4 task).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "results" / "week4"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "results" / "per_image_results.csv")
cfg = yaml.safe_load(open(ROOT / "configs" / "eval.yaml"))
size_cfg = cfg["stratification"]["polyp_size_pct"]
METRICS = ["dice", "iou", "boundary_iou", "hd95"]
print(df.shape, df["arm"].value_counts().to_dict())

(3400, 10) {'unet': 2400, 'sam3': 1000}


## Sanity checks on the frozen file

In [2]:
assert len(df) == 3400, len(df)
assert df.groupby("image_id")["gt_area_pct"].nunique().max() == 1   # area depends on the image only
assert df.duplicated(["arm", "run_id", "image_id"]).sum() == 0
assert df["image_id"].nunique() == 200
print("ok: 3,400 rows, 200 images, no duplicates")

ok: 3,400 rows, 200 images, no duplicates


## Task 1.1 — assign each test image to a size bin

In [3]:
LABELS = ["small", "medium", "large"]
edges = [0, size_cfg["small"][1], size_cfg["medium"][1], np.inf]      # [0,5) [5,15) [15,inf)
df["size_bin"] = pd.cut(df["gt_area_pct"], bins=edges, labels=LABELS, right=False)
assert df["size_bin"].notna().all()

assign = (df.drop_duplicates("image_id")[["image_id", "gt_area_pct", "size_bin"]]
            .sort_values("gt_area_pct").reset_index(drop=True))
assign.to_csv(OUT / "size_bin_assignment.csv", index=False)

n_bin = assign["size_bin"].value_counts().reindex(LABELS)
print(n_bin.to_frame("n_images"))
print(assign.groupby("size_bin", observed=True)["gt_area_pct"].agg(["min", "median", "max"]).round(2))

          n_images
size_bin          
small           39
medium          81
large           80
            min  median    max
size_bin                      
small      0.53    3.48   4.95
medium     5.01    9.82  14.98
large     15.50   25.26  71.87


## Task 1.3 — build the stratified tables

Columns are the *conditions*: four U-Net brackets and five SAM 3 prompts.

In [4]:
df["bracket"] = df["run_id"].str.extract(r"^(\d+)pct_seed\d+$")[0]
df["condition"] = np.where(df["arm"] == "unet",
                           "unet-" + df["bracket"].fillna("") + "%",
                           "sam3-" + df["run_id"])
ORDER = ["unet-10%", "unet-25%", "unet-50%", "unet-100%",
         "sam3-box-gt", "sam3-box-j10", "sam3-box-j20", "sam3-pt-1", "sam3-pt-3"]
assert set(df["condition"]) == set(ORDER), set(df["condition"]) ^ set(ORDER)

# HD95 is inf where the model predicted an empty mask (Dice = 0). Exclude from HD95 means, and count them.
n_inf = np.isinf(df["hd95"]).sum()
excluded = (df[np.isinf(df["hd95"])].groupby(["condition", "size_bin"], observed=True).size()
              .unstack("size_bin").reindex(index=ORDER, columns=LABELS).fillna(0).astype(int))
excluded.to_csv(OUT / "hd95_excluded_by_bin.csv")
df["hd95"] = df["hd95"].replace([np.inf, -np.inf], np.nan)
assert excluded.values.sum() == n_inf
print("non-finite HD95 rows excluded:", n_inf)
excluded

non-finite HD95 rows excluded: 19


size_bin,small,medium,large
condition,,,
unet-10%,3,0,0
unet-25%,2,0,0
unet-50%,7,0,0
unet-100%,7,0,0
sam3-box-gt,0,0,0
sam3-box-j10,0,0,0
sam3-box-j20,0,0,0
sam3-pt-1,0,0,0
sam3-pt-3,0,0,0


In [5]:
# 1) mean over images within each run and bin; 2) mean and std over seeds (SAM 3 has one run -> std is NaN)
run_means = df.groupby(["condition", "run_id", "size_bin"], observed=True)[METRICS].mean()
summary = (run_means.groupby(["condition", "size_bin"], observed=True)
                    .agg(["mean", "std"]))
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()
summary["n_images"] = summary["size_bin"].map(n_bin).astype(int)
summary["condition"] = pd.Categorical(summary["condition"], ORDER, ordered=True)
summary["size_bin"] = pd.Categorical(summary["size_bin"], LABELS, ordered=True)
summary = summary.sort_values(["size_bin", "condition"]).reset_index(drop=True)
summary.to_csv(OUT / "size_stratified_summary.csv", index=False)
summary.head(9)

,condition,size_bin,dice_mean,dice_std,iou_mean,iou_std,boundary_iou_mean,boundary_iou_std,hd95_mean,hd95_std,n_images
0,unet-10%,small,0.584423,0.012329,0.482962,0.004519,0.357460,0.001729,152.603789,6.564949,39
1,unet-25%,small,0.652730,0.018939,0.557792,0.012759,0.425363,0.020467,118.553663,4.200294,39
2,unet-50%,small,0.698176,0.027258,0.614833,0.028259,0.485916,0.028060,107.456102,13.841251,39
3,unet-100%,small,0.720923,0.021854,0.638484,0.022257,0.503160,0.023846,99.670465,5.754930,39
4,sam3-box-gt,small,0.939348,NaN,0.886628,NaN,0.749727,NaN,6.658945,NaN,39
5,sam3-box-j10,small,0.935588,NaN,0.880215,NaN,0.740877,NaN,7.278630,NaN,39
6,sam3-box-j20,small,0.901956,NaN,0.825279,NaN,0.642776,NaN,13.009168,NaN,39
7,sam3-pt-1,small,0.781333,NaN,0.703736,NaN,0.552564,NaN,39.320750,NaN,39
8,sam3-pt-3,small,0.865280,NaN,0.801840,NaN,0.651676,NaN,31.300046,NaN,39


In [6]:
def table(metric, digits=3):
    """rows = size bins, columns = conditions, cells = mean (± seed std for U-Net)."""
    m = summary.pivot(index="size_bin", columns="condition", values=f"{metric}_mean")
    s = summary.pivot(index="size_bin", columns="condition", values=f"{metric}_std")
    fmt = lambda a, b: f"{a:.{digits}f}" if np.isnan(b) else f"{a:.{digits}f} ± {b:.{digits}f}"
    return pd.DataFrame({c: [fmt(m.loc[r, c], s.loc[r, c]) for r in LABELS] for c in ORDER}, index=LABELS)

for metric in METRICS:
    print(f"\n=== {metric} ===")
    display(table(metric, digits=1 if metric == "hd95" else 3))


=== dice ===


,unet-10%,unet-25%,unet-50%,unet-100%,sam3-box-gt,sam3-box-j10,sam3-box-j20,sam3-pt-1,sam3-pt-3
small,0.584 ± 0.012,0.653 ± 0.019,0.698 ± 0.027,0.721 ± 0.022,0.939,0.936,0.902,0.781,0.865
medium,0.833 ± 0.003,0.859 ± 0.011,0.879 ± 0.004,0.890 ± 0.009,0.926,0.899,0.823,0.877,0.901
large,0.792 ± 0.003,0.826 ± 0.018,0.843 ± 0.009,0.862 ± 0.002,0.840,0.748,0.587,0.781,0.829



=== iou ===


,unet-10%,unet-25%,unet-50%,unet-100%,sam3-box-gt,sam3-box-j10,sam3-box-j20,sam3-pt-1,sam3-pt-3
small,0.483 ± 0.005,0.558 ± 0.013,0.615 ± 0.028,0.638 ± 0.022,0.887,0.880,0.825,0.704,0.802
medium,0.739 ± 0.006,0.776 ± 0.015,0.802 ± 0.006,0.818 ± 0.012,0.878,0.841,0.749,0.816,0.840
large,0.682 ± 0.003,0.726 ± 0.023,0.750 ± 0.012,0.774 ± 0.003,0.770,0.669,0.490,0.694,0.741



=== boundary_iou ===


,unet-10%,unet-25%,unet-50%,unet-100%,sam3-box-gt,sam3-box-j10,sam3-box-j20,sam3-pt-1,sam3-pt-3
small,0.357 ± 0.002,0.425 ± 0.020,0.486 ± 0.028,0.503 ± 0.024,0.750,0.741,0.643,0.553,0.652
medium,0.497 ± 0.007,0.534 ± 0.016,0.568 ± 0.007,0.579 ± 0.015,0.655,0.583,0.478,0.585,0.597
large,0.348 ± 0.006,0.390 ± 0.025,0.423 ± 0.008,0.434 ± 0.002,0.441,0.330,0.221,0.380,0.393



=== hd95 ===


,unet-10%,unet-25%,unet-50%,unet-100%,sam3-box-gt,sam3-box-j10,sam3-box-j20,sam3-pt-1,sam3-pt-3
small,152.6 ± 6.6,118.6 ± 4.2,107.5 ± 13.8,99.7 ± 5.8,6.7,7.3,13.0,39.3,31.3
medium,88.7 ± 6.6,78.4 ± 3.3,77.5 ± 2.4,61.6 ± 4.6,18.1,23.5,34.4,39.2,32.0
large,103.8 ± 3.1,86.5 ± 2.1,81.6 ± 7.1,68.1 ± 3.2,65.1,67.7,92.4,95.2,81.3


## Check: stratified means must reproduce the overall means

In [7]:
# Every run scores all 200 images, so the count-weighted average of the three bin means
# must equal the run's overall mean (Dice shown; do the same for IoU/Boundary IoU if you like).
by_bin = run_means["dice"].unstack("size_bin")[LABELS]
weighted = by_bin.mul(n_bin[LABELS]).sum(axis=1) / n_bin.sum()
direct = df.groupby(["condition", "run_id"])["dice"].mean()
assert np.allclose(weighted.sort_index().values, direct.reindex(weighted.index).sort_index().values)
print("ok: bin-weighted Dice == overall Dice for all 17 runs")

ok: bin-weighted Dice == overall Dice for all 17 runs


## Next
- **Task 1.4:** compare these tables with the prior expectation (SAM 3 relatively better on large, well-defined polyps) and write your 2–3 sentence note in `decisions.md`. Say what you see, including if it contradicts the expectation.
- HD95 cells exclude empty predictions; quote `hd95_excluded_by_bin.csv` next to any HD95 number you report.
- Small bin has only ~39 images; say so when you interpret it.

In [8]:
UNETS = ["unet-10%", "unet-25%", "unet-50%", "unet-100%"]

def gap_table(metric, sam="sam3-box-gt"):
    """SAM 3 minus U-Net, per size bin. For Dice/IoU/Boundary IoU, positive = SAM 3 ahead.
    For HD95 (lower is better), NEGATIVE = SAM 3 ahead."""
    m = summary.pivot(index="size_bin", columns="condition", values=f"{metric}_mean")
    g = pd.DataFrame({u: m[sam] - m[u] for u in UNETS}).loc[LABELS]
    g.loc["large minus small"] = g.loc["large"] - g.loc["small"]
    return g

for metric in ["dice", "boundary_iou", "hd95"]:
    for sam in ["sam3-box-gt", "sam3-pt-1"]:
        print(f"\n=== {metric}: {sam} minus U-Net ===")
        display(gap_table(metric, sam).round(3))

# How noisy is U-Net from seed to seed? Compare gap differences against this.
display(summary.pivot(index="size_bin", columns="condition", values="dice_std")[UNETS].loc[LABELS].round(3))


=== dice: sam3-box-gt minus U-Net ===


,unet-10%,unet-25%,unet-50%,unet-100%
size_bin,,,,
small,0.355,0.287,0.241,0.218
medium,0.093,0.067,0.047,0.036
large,0.048,0.015,-0.003,-0.022
large minus small,-0.307,-0.272,-0.244,-0.240



=== dice: sam3-pt-1 minus U-Net ===


,unet-10%,unet-25%,unet-50%,unet-100%
size_bin,,,,
small,0.197,0.129,0.083,0.060
medium,0.044,0.018,-0.002,-0.013
large,-0.011,-0.045,-0.062,-0.081
large minus small,-0.208,-0.173,-0.145,-0.142



=== boundary_iou: sam3-box-gt minus U-Net ===


...


=== hd95: sam3-box-gt minus U-Net ===


,unet-10%,unet-25%,unet-50%,unet-100%
size_bin,,,,
small,-145.945,-111.895,-100.797,-93.012
medium,-70.543,-60.329,-59.364,-43.467
large,-38.698,-21.394,-16.426,-2.982
large minus small,107.247,90.500,84.371,90.029



=== hd95: sam3-pt-1 minus U-Net ===


,unet-10%,unet-25%,unet-50%,unet-100%
size_bin,,,,
small,-113.283,-79.233,-68.135,-60.350
medium,-49.494,-39.280,-38.315,-22.418
large,-8.693,8.611,13.578,27.023
large minus small,104.590,87.843,81.714,87.372


condition,unet-10%,unet-25%,unet-50%,unet-100%
size_bin,,,,
small,0.012,0.019,0.027,0.022
medium,0.003,0.011,0.004,0.009
large,0.003,0.018,0.009,0.002
